In [ ]:
import os
from pathlib import Path
import shutil

import pandas as pd
import functools
import numpy as np

import teehr

import pyspark as ps
from pyspark.sql import functions as F


teehr.__version__

### Setup

In [ ]:
test_eval_dir = Path(Path().cwd(), "FIRO_test_evaluation")

# test updating numba cache
conf_update = {"spark.executorEnv.NUMBA_CACHE_DIR":"/tmp/numba_cache"}

# spark configuration
from teehr.evaluation.spark_session_utils import create_spark_session
spark = create_spark_session(
    start_spark_cluster=True,
    executor_instances=2,
    executor_memory="50g",
    executor_cores=7,
    update_configs=conf_update
)

# access local evaluation
ev = teehr.LocalReadWriteEvaluation(
    dir_path=test_eval_dir,
    spark=spark
)

ev.enable_logging()

In [ ]:
# Configure bootstrap
from teehr.metrics.models.bootstrap import Bootstrappers

boot = Bootstrappers.CircularBlock(
    reps=1000,
    # block_size=None -> auto estimate using optimal_block_length (b_cb)
    block_size=None,
    seed=42,
    quantiles=[0.05, 0.5, 0.95]
)

### Location Analysis
---------------------------------------------------
- Global Filters/Toggles:
  - Model (configuration) -> simulated
  - Baseline (climatology, another model)
  - Decision Variable

#### Location Analysis: Deterministic Assessment
---------------------------------------------------
- Filters/Toggles:
  - Mean or median deterministic forecast
  - Show uncertainty (boolean)
- Filters:
  - Season
  - Observed flow quantile
---------------------------------------------------
- Continuous Metrics:
  - MAE vs lead time
  - RMSE vs lead time
  - Bias vs lead time
  - Correlation vs lead time
  - Normalized RMSE vs lead time (MISSING)
  - NSE vs lead time
- Event Threshold Metrics:
  - POD vs lead time
  - FAR vs lead time
  - CSI vs lead time
  - Frequency Bias vs lead time
- Diagnostic Plots:
  - Performance diagram
  - Contingency Table        

In [ ]:
def create_continuous_table(
    event_column: str | None,
    configuration_toggle: str,
    variable_toggle: str,
    seasons: bool,
) -> ps.sql.DataFrame:
    '''Creates continuous metric table with and without quantiles considered'''
    
    print(f"Beginning calculations for event_column = {event_column}")

    # define group_by_list
    group_by_list = [
        "primary_location_id",
        "configuration_name",
        "variable_name",
        "season",
        "forecast_lead_time",
        "forecast_lead_time_bin"
    ]
    if not seasons:
        group_by_list.remove("season")

    # define metrics_list
    metrics_list = [
        teehr.DeterministicMetrics.MeanAbsoluteError(),
        teehr.DeterministicMetrics.RootMeanSquareError(),
        teehr.DeterministicMetrics.RelativeBias(),
        teehr.DeterministicMetrics.PearsonCorrelation(),
        teehr.DeterministicMetrics.NashSutcliffeEfficiency()
    ]

    # define filter_list
    formatted_configs = ",".join([f"'{x}'" for x in configuration_toggle])
    filter_list = [
         f"configuration_name IN ({formatted_configs})",
         f"variable_name = '{variable_toggle}'",   
    ]
    if event_column:
        filter_list.append(f"{event_column} = true")
    
    # perform aggregation
    sdf = ev.table(
        'joined_timeseries'
    ).filter(
        filter_list 
    ).aggregate(
        metrics = metrics_list,
        group_by = group_by_list
    ).order_by(
        [
        "primary_location_id", 
        "forecast_lead_time"
        ]
    ).to_sdf()

    # handle event_column
    if event_column:
        sdf = sdf.withColumn("threshold", F.lit(f"q_{event_column.split('_')[-1]}"))
    else:
        sdf = sdf.withColumn("threshold", F.lit(None))

    # handle season column
    if not seasons:
        # df['season'] = None
        sdf = sdf.withColumn("season", F.lit(None))

    print(f"Finished calculations for event_column = {event_column}")
    
    return sdf


In [ ]:
def create_threshold_table(
    threshold_column: str,
    configuration_toggle: str,
    variable_toggle: str,
    seasons: bool
) -> ps.sql.DataFrame:
    '''Creates one of several threshold-based metrics tables'''

    print(f"Beginning calculations for threshold_column = {threshold_column}\n")

    # define group_by lists 
    group_by_list= [
        "primary_location_id",
        "configuration_name",
        "variable_name",
        "season",
        "forecast_lead_time",
        "forecast_lead_time_bin"
    ]
    if not seasons:
        group_by_list.remove('season')

    # define metrics lists
    metrics_list = [
        teehr.DeterministicMetrics.ConfusionMatrix(threshold_field_name=threshold_column, unpack_results=True),
        teehr.DeterministicMetrics.ProbabilityOfDetection(threshold_field_name=threshold_column),
        teehr.DeterministicMetrics.FalseAlarmRatio(threshold_field_name=threshold_column),
        teehr.DeterministicMetrics.CriticalSuccessIndex(threshold_field_name=threshold_column),
        teehr.DeterministicMetrics.FrequencyBiasIndex(threshold_field_name=threshold_column)
    ]

    # define filters list
    formatted_configs = ",".join([f"'{x}'" for x in configuration_toggle])
    filter_list = [
         f"configuration_name IN ({formatted_configs})",
         f"variable_name = '{variable_toggle}'",   
    ]

    # perform aggregation
    sdf = ev.table(
        "joined_timeseries"
    ).filter(
        filter_list
    ).aggregate(
        metrics = metrics_list,
        group_by = group_by_list
    ).order_by(
        [
        "primary_location_id", 
        "forecast_lead_time"
        ]
    ).to_sdf()

    # handle threshold column
    sdf = sdf.withColumn("threshold", F.lit(f'{threshold_column}'))

    # handle season column
    if not seasons:
        sdf = sdf.withColumn("season", F.lit(None))

    print(f"Finished calculations for threshold_column = {threshold_column}\n")

    return sdf


In [ ]:
# define result dict
result = {}

In [ ]:
# assemble continuous table with no seasons
print('\n Assembling continuous table with no seasons ... \n')

event_cols = [None, 'event_10th', 'event_25th', 'event_50th', 'event_75th', 'event_90th']
configuration_toggle = ["hefs_streamflow_forecast", "benchmark_forecast_hourly_normals"]
variable_toggle = "streamflow_hourly_inst"
seasons = False

continuous_no_szn = {}
for event_col in event_cols:
    if event_col:
        continuous_no_szn[event_col] = create_continuous_table(
            event_column=event_col,
            configuration_toggle=configuration_toggle,
            variable_toggle=variable_toggle,
            seasons=seasons
        )
    else:
        continuous_no_szn['None'] = create_continuous_table(
            event_column=event_col,
            configuration_toggle=configuration_toggle,
            variable_toggle=variable_toggle,
            seasons=seasons
        )

continuous_no_szns_sdf = functools.reduce(ps.sql.DataFrame.unionByName, continuous_no_szn.values())
result['continuous_no_szn'] = continuous_no_szns_sdf

# assemble continuous table with seasons
print('\n Assembling continuous table with seasons ... \n')

seasons = True

continuous_w_szn = {}
for event_col in event_cols:
    if event_col:
        continuous_w_szn[event_col] = create_continuous_table(
            event_column=event_col,
            configuration_toggle=configuration_toggle,
            variable_toggle=variable_toggle,
            seasons=seasons
        )
    else:
        continuous_w_szn['None'] = create_continuous_table(
            event_column=event_col,
            configuration_toggle=configuration_toggle,
            variable_toggle=variable_toggle,
            seasons=seasons
        )

continuous_w_szns_sdf = functools.reduce(ps.sql.DataFrame.unionByName, continuous_w_szn.values())
result['continuous_w_szn'] = continuous_w_szns_sdf

# assemble threshold table with no seasons
print('\n Assembling thresholds table with no seasons ... \n')

thresholds = ['q_10th', 'q_25th', 'q_50th', 'q_75th', 'q_90th']
configuration_toggle = ["hefs_streamflow_forecast", "benchmark_forecast_hourly_normals"]
variable_toggle = "streamflow_hourly_inst"
seasons = False

thresholds_no_szn = {}
for threshold in thresholds:
    thresholds_no_szn[threshold] = create_threshold_table(
        threshold_column=threshold, 
        configuration_toggle=configuration_toggle, 
        variable_toggle=variable_toggle, 
        seasons=seasons
    )

thresholds_no_szns_sdf = functools.reduce(ps.sql.DataFrame.unionByName, thresholds_no_szn.values())
result['threshold_no_szn'] = thresholds_no_szns_sdf

# assemble threshold table with seasons
print('\n Assembling thresholds table with seasons ... \n')

seasons = True

thresholds_w_szn = {}
for threshold in thresholds:
    thresholds_w_szn[threshold] = create_threshold_table(
        threshold_column=threshold, 
        configuration_toggle=configuration_toggle, 
        variable_toggle=variable_toggle, 
        seasons=seasons
    )

thresholds_w_szns_sdf = functools.reduce(ps.sql.DataFrame.unionByName, thresholds_w_szn.values())
result['threshold_w_szn'] = thresholds_w_szns_sdf

#### Location Analysis: Full Distribution Metrics
---------------------------------------------------
- Filters/Toggles:
  - Show uncertainty (boolean)
- Filters:
  - Season
  - Observed flow quantile
---------------------------------------------------
- Lead time summary:
  - CRPSS vs lead time
  - CRPS vs lead time   

In [ ]:
def create_crps_table(
    event_column: str | None,
    configuration_toggle: str,
    variable_toggle: str,
    seasons: bool
) -> ps.sql.DataFrame:
    '''Creates CRPSS table for all flow thresholds, with or without seasons'''

    print(f"Beginning calculations for event_column = {event_column}\n")

    # define group_by list
    group_by_list = [
        "primary_location_id",
        "configuration_name",
        "variable_name",
        "season",
        "forecast_lead_time",
        "forecast_lead_time_bin"
    ]
    if not seasons:
        group_by_list.remove('season')

    # define metrics list
    crps = teehr.ProbabilisticMetrics.CRPS()
    crps.summary_func = np.mean
    crps.estimator = "pwm"
    crps.backend = "numba"
    crps.reference_configuration = "benchmark_forecast_hourly_normals"
    metrics_list = [crps]

    # define filters list
    formatted_configs = ",".join([f"'{x}'" for x in configuration_toggle])
    filter_list = [
         f"configuration_name IN ({formatted_configs})",
         f"variable_name = '{variable_toggle}'",
    ]

    if event_column:
        filter_list.append(f"{event_column} = true")

    # perform aggregation
    sdf = ev.table(
        "joined_timeseries"
    ).filter(
        filter_list
    ).aggregate(
        metrics = metrics_list,
        group_by = group_by_list
    ).order_by(
        [
        "primary_location_id", 
        "forecast_lead_time"
        ]
    ).to_sdf()

    # handle threshold column
    if event_column:
        sdf = sdf.withColumn("threshold", F.lit(f"q_{event_column.split('_')[-1]}"))
    else:
        sdf = sdf.withColumn("threshold", F.lit(None))

    # handle season column
    if not seasons:
        sdf = sdf.withColumn("season", F.lit(None))

    print(f"Finished calculations for event_column = {event_column}\n")

    return sdf

In [ ]:
# assemble crps table w/ no seasons
print('\n Assembling CRPS table with no seasons ... \n')

event_cols = [None, 'event_10th', 'event_25th', 'event_50th', 'event_75th', 'event_90th']
configuration_toggle = ["hefs_streamflow_forecast", "benchmark_forecast_hourly_normals"]
variable_toggle = "streamflow_hourly_inst"
seasons = False

crps_no_szn = {}
for event in event_cols:
    if event:
        crps_no_szn[event] = create_crps_table(
            event_column=event, 
            configuration_toggle=configuration_toggle,
            variable_toggle=variable_toggle,
            seasons=seasons
        )
    else:
        crps_no_szn['None'] = create_crps_table(
            event_column=event, 
            configuration_toggle=configuration_toggle,
            variable_toggle=variable_toggle,
            seasons=seasons
        )

crps_no_szns_sdf = functools.reduce(ps.sql.DataFrame.unionByName, crps_no_szn.values())
result['crps_no_szn'] = crps_no_szns_sdf

# assemble crps table w/ seasons
print('\n Assembling CRPS table with seasons ... \n')

seasons = True

crps_w_szn = {}
for event in event_cols:
    if event:
        crps_w_szn[event] = create_crps_table(
            event_column=event, 
            configuration_toggle=configuration_toggle,
            variable_toggle=variable_toggle,
            seasons=seasons
        )
    else:
        crps_w_szn['None'] = create_crps_table(
            event_column=event, 
            configuration_toggle=configuration_toggle,
            variable_toggle=variable_toggle,
            seasons=seasons
        )

crps_w_szns_sdf = functools.reduce(ps.sql.DataFrame.unionByName, crps_w_szn.values())
result['crps_w_szn'] = crps_w_szns_sdf

#### Location Analysis: Event Threshold Metrics
---------------------------------------------------
- Filters/Toggles:
  - Show uncertainty (boolean)
---------------------------------------------------
- Event threshold metrics:
  - BSS vs lead time
  - BS vs lead time
  - BS decomposition vs lead time (pending)
- Diagrams:
  - Reliability diagram + sharpness histogram
  - ROC Curve + Discrimination diagram 

In [ ]:
def create_bss_table(
    threshold_col: str,
    threshold_val: str,
    configuration_toggle: str,
    variable_toggle: str,
    seasons: bool
) -> ps.sql.DataFrame:
    '''Creates BSS table for all flow thresholds, with or without seasons'''
    
    print(f"Beginning calculations for threshold = {threshold_col}\n")

    # define group_by list
    group_by_list = [
        "primary_location_id",
        "configuration_name",
        "variable_name",
        "season",
        "forecast_lead_time",
        "forecast_lead_time_bin"
    ]
    if not seasons:
        group_by_list.remove('season')
        
    # define metrics list
    bs = teehr.ProbabilisticMetrics.BrierScore()
    bs.threshold = threshold_val
    bs.summary_func = np.mean
    bs.backend = "numba"
    bs.reference_configuration = "benchmark_forecast_hourly_normals"
    metrics_list = [bs]

    # define filters list
    formatted_configs = ",".join([f"'{x}'" for x in configuration_toggle])
    filter_list = [
         f"configuration_name IN ({formatted_configs})",
         f"variable_name = '{variable_toggle}'",   
    ]
    
    # perform aggregation
    sdf = ev.table(
        "joined_timeseries"
    ).filter(
        filter_list
    ).aggregate(
        metrics=metrics_list,
        group_by=group_by_list,
    ).order_by(
            [
            "primary_location_id", 
            "forecast_lead_time"
            ]
    ).to_sdf()
    
    # handle threshold column
    sdf = sdf.withColumn('threshold', F.lit(f'{threshold_col}'))

    # handle season column
    if not seasons:
        sdf = sdf.withColumn('season', F.lit(None))

    print(f"Finished calculations for event_column = {threshold_col}\n")

    return sdf

In [ ]:
# assemble bss table w/ no seasons
print('\n Assembling BSS table with no seasons ... \n')

threshold_dict = {'q_10th':0.10,
                  'q_25th':0.25,
                  'q_50th':0.50,
                  'q_75th':0.75,
                  'q_90th':0.90}
configuration_toggle = ["hefs_streamflow_forecast", "benchmark_forecast_hourly_normals"]
variable_toggle = "streamflow_hourly_inst"
seasons = False

bss_no_szn = {}

for key in threshold_dict.keys():
    threshold_col = key
    threshold_val = threshold_dict[key]
    bss_no_szn[key] = create_bss_table(
        threshold_col=threshold_col, 
        threshold_val=threshold_val,
        configuration_toggle=configuration_toggle,
        variable_toggle=variable_toggle,
        seasons=seasons
    )

bss_no_szns_sdf = functools.reduce(ps.sql.DataFrame.unionByName, bss_no_szn.values())
result['bss_no_szn'] = bss_no_szns_sdf

# assemble bss table w/ seasons
print('\n Assembling BSS table with seasons ... \n')

seasons = True

bss_w_szn = {}

for key in threshold_dict.keys():
    threshold_col = key
    threshold_val = threshold_dict[key]
    bss_w_szn[key] = create_bss_table(
        threshold_col=threshold_col, 
        threshold_val=threshold_val,
        configuration_toggle=configuration_toggle,
        variable_toggle=variable_toggle,
        seasons=seasons
    )

bss_w_szns_sdf = functools.reduce(ps.sql.DataFrame.unionByName, bss_w_szn.values())
result['bss_w_szn'] = bss_w_szns_sdf

### Assemble result and write

In [ ]:
# Common columns to merge on
common_cols = [
    "primary_location_id",
    "configuration_name", 
    "variable_name",
    "season",
    "forecast_lead_time",
    "forecast_lead_time_bin",
    "threshold"
]

# define groupings to reduce and join
locations_no_szns = [result['continuous_no_szn'], result['threshold_no_szn'], result['crps_no_szn'], result['bss_no_szn']]
locations_w_szns = [result['continuous_w_szn'], result['threshold_w_szn'], result['crps_w_szn'], result['bss_w_szn']]

# reduce and join like tables
locations_no_szns_sdf = functools.reduce(
    lambda left, right: left.join(right, on=common_cols, how='outer'),
    locations_no_szns
)
locations_w_szns_sdf = functools.reduce(
    lambda left, right: left.join(right, on=common_cols, how='outer'),
    locations_w_szns
)

# concatenate unlike tables
locations_sdf = locations_no_szns_sdf.unionByName(locations_w_szns_sdf, allowMissingColumns=True)

In [ ]:
# write to warehouse
ev._write.to_warehouse(source_data=locations_sdf, table_name='locations_metrics', write_mode='create_or_replace')

### Kill Spark

In [ ]:
ev.spark.stop()